In [1]:
#Import the Necessary Libraries
import sympy as sm 
import sympy.physics.mechanics as me 
me.init_vprinting(use_latex="mathjax")
import numpy as np
import scipy as sp 
import scipy.optimize as so 
import scipy.integrate as si
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib as litt
litt.rcParams['animation.embed_limit'] = 50.0

#Importing ipywidgets for Visualizations
import ipywidgets as iw
from ipywidgets import interact

In [2]:
#Setting the variables and Reference Frames and points
N=me.ReferenceFrame("N")
A=me.ReferenceFrame("A")
B=me.ReferenceFrame("B")
q1, q2=me.dynamicsymbols("q_1,q_2")
l1, l2=sm.symbols("l_1, l_2")

In [3]:
#Define the respt. points
o=me.Point("O")
p1=me.Point("P1")
p2=me.Point("P2")
A.orient_axis(N, N.z, q1)
B.orient_axis(A, A.z, q2)
p1.set_pos(o, l1*A.x)
p2.set_pos(p1, l2*B.x)

#Initialize the end effector positions w.r.t base FOR
x_e, y_e=me.dynamicsymbols("x_e, y_e")

In [4]:
#Initialize the matrices 
q_m=sm.Matrix([q1, q2])
q_m_d=sm.Matrix([q1.diff(), q2.diff()])
s=[l1, l2]
xy_e=sm.Matrix([x_e, y_e])

In [5]:
#Setting the vectors
r_p1_o=p1.pos_from(o)
r_p2_p1=p2.pos_from(p1)

r_p2_o=x_e*N.x+y_e*N.y
pos=r_p1_o+r_p2_p1-r_p2_o
pos

l₁ a_x + l₂ b_x + -xₑ n_x + -yₑ n_y

In [6]:
#Formulate the holonomic constraint matrix-IK
f_h=sm.simplify(sm.Matrix([[pos.dot(N.x)], [pos.dot(N.y)]]))
f_h

⎡l₁⋅cos(q₁) + l₂⋅cos(q₁ + q₂) - xₑ⎤
⎢                                 ⎥
⎣l₁⋅sin(q₁) + l₂⋅sin(q₁ + q₂) - yₑ⎦

# Idea of Inverse Kinematics (Without Explicitly using Jacobians)-P2P IK

The modelling proposed sometimes is also called a P2P IK. 
In Forward Kinemtics, our goal is to **obtain the position of the end effector in space** as the angle of the joints, change w.r.t space. In Inverse Kinematics, our idea is to **find the angle of joints ($q_1, q_2$)** for a **given end-effector position**. 
Hence we have to solve $f_h$. Which is: 
$$
f_h(q_1, q_2)=0
$$ 
Interestingly, we are **bypassing** the calclations of Jacobians **Explicitly** (as we dont define ANY Jacobians anywhere) using Numerical Iterative Techniques here primarily because we obtain the respective position of the EF. However ```fsolve()``` uses a default method which is **hybrj** which essentially uses **Jacobians** to let the values converge faster. And given that $f_h$ is a *Transcendental Equation* its roots have to be solved using Numerical Root Finding Techniques.  

In [7]:
#Conversion from Sympy to Numpy Expressions and data type
f_s=sm.lambdify([q_m, xy_e, s], f_h)

#Assign the guess values
q_mg=np.array([np.pi*(-50)/180, np.pi*(30)/180])          #Guess Values for Joint Angles- -50deg, -3deg
xy_es=np.array([1.0, -1.0])                             #Actual End Effector positions
p=np.array([1, 1.2])

# Solutions of Transcendental equations

Using Scientific Python (SciPy) one can make use of ```fsolve()``` to solve simple transcendental equations. By the definition, fsolve() requires an input function which is defined as ```func_sol(v_sol, v1, v2....)``` where, ```v_sol``` is the required variables attained Numerically. First ```v_sol``` must be declared. 

Under the hood, ```fsolve()``` uses a **Modified Version of Newton-Raphson** technique (which sometimes is also called as **Powell's Hybrid Technique** which essentially is a combination of **Newton Raphson** (used when the solution is closer to the root) and **Gradient Descent Step** (used when the solution is far away from the root)). To facilitate it, the algorithm uses **Jacobians** implicitly. 

In [8]:
def sol(q_mg, xy_es, p): 
    f_so=np.squeeze(f_s(q_mg, xy_es, p))
    return f_so
f_sol=so.fsolve(sol, q_mg, args=(xy_es,p))
np.rad2deg(f_sol)

array([-101.52704638,  100.56397759])

# Considerations and stability

Now, if we use the same technique for some point say $(x_e, y_e)=(3,-1)$, the solver would inherently **fail**. Because the **max end effector position** when both link-1, link-2 are aligned (i.e when $q_2=0$) is: 
$$
l_1+l_2=\text{max end effector position}=1+1.2=2.2m
$$
And so, physically it'd be illogical for the links to reach anywhere beyond it. So the limits of $x_e, y_e$ can be determined as:
$$
\sqrt{x_e^2+y_e^2}=2.2=\phi(x,y)
$$
Using Parameterizations we can say: 
$$
x_e=2.2*\cos(\theta)\\
y_e=2.2*\sin(\theta)\\
$$
And substituting in $\phi(x,y)$: 
$$
\cos^2(\theta)+\sin^2(\theta)=1
$$
Already we can say when $\theta=0$, we can say that $x_e=2.2$ and $y_e=0$. (Which is one possible value). The vice-versa would be flipping both the $x_e$ and $y_e$. The other values have to be attained by differentiation for max($\theta$) in $\phi$. So, its obvious the max possible value over $\theta \in [0,\frac{\pi}{2}]$ is: 
$$ 
\theta=\frac{\pi}{4}
$$
And hence, $x_e=|\frac{2.2}{\sqrt{2}}|\approx|1.566|m$ and $y_e=|\frac{2.2}{\sqrt{2}}|\approx|1.566|m$ over the entire $\theta \in [0, 2\pi]$. Long story short, the values of $x_e$, $y_e$ have to be such that: 
$$
x_e^2+y_e^2 \leq (l_1+l_2)^2
$$

In [9]:
# Solving for a range of values of x_e, y_e
xy_el=np.array([np.linspace(-1, 1, 10), np.linspace(-1, 1, 10)])

def res(xy_el, p): 
    res=[]
    for i in range(10): 
        solv=np.array([xy_el[0, i], xy_el[1,i]])
        if solv[0]>0 and solv[1]<0: 
            q_mg=np.array([np.deg2rad(-5), np.deg2rad(-45)])
        elif solv[0]>0 and solv[1]>0: 
            q_mg=np.array([np.deg2rad(10), np.deg2rad(-20)])
        elif solv[0]<0 and solv[1]>0: 
            q_mg=np.array([np.deg2rad(100), np.deg2rad(-60)])
        else:
            q_mg=np.array([np.deg2rad(-100), np.deg2rad(-200)])
        q_1a, q_1b=so.fsolve(sol, q_mg, args=(solv, p))
        res+=[[q_1a, q_1b]]
    return np.rad2deg(np.array([res]))
t=res(xy_el, p)

/tmp/ipykernel_37385/3507736644.py:16: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last five Jacobian evaluations.
  q_1a, q_1b=so.fsolve(sol, q_mg, args=(solv, p))


In [16]:
#Visualization
points=[p1, p2]
coord=sm.Matrix(o.pos_from(o).to_matrix(N))
for i in points: 
    coord=coord.row_join(i.pos_from(o).to_matrix(N))
e_c=sm.lambdify([xy_e, s, q_m], coord)

@interact(x_e=(-1,1,0.2), y_e=(-1,1,0.2))
def vis(x_e, y_e): 
    fig, axes=plt.subplots()
    sol1=np.array([x_e, y_e])
    global p
    if x_e>=0 and y_e<=0: 
            q_mg=np.array([np.deg2rad(-5), np.deg2rad(-45)])
    elif x_e>0 and y_e>0: 
            q_mg=np.array([np.deg2rad(10), np.deg2rad(-20)])
    elif x_e<=0 and y_e>=0: 
            q_mg=np.array([np.deg2rad(100), np.deg2rad(-60)])
    elif x_e==0 and y_e==0: 
        pass   
    else:
            q_mg=np.array([np.deg2rad(-100), np.deg2rad(-200)])
    res=so.fsolve(sol, q_mg, args=(sol1, p))
    res=res%(2*np.pi)           #Force the angles to the domain [0,2pi]
    x_p, y_p, z_p=e_c(sol1, p, res)
    fig.set_size_inches(10,10)
    axes.set_aspect("equal")
    axes.grid()
    lines, = axes.plot(x_p, y_p, color='black',
                     marker='o', markerfacecolor='red', markersize=10)
    title_text = axes.set_title("2R Inverse Kinematics")
    text = axes.text(0, 0, '', fontsize=10, 
               bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.6, ec="orange"))
    text.set_position((2, 2))
    text.set_text(f"$q_1$: {np.rad2deg(res[0]):.2f}\n$q_2$: {np.rad2deg(res[1]):.2f}")
    axes.set_xlim((-2.0, 3.0))
    axes.set_ylim((-2.0, 1.5))
    axes.set_xlabel('$x$ [m]')
    axes.set_ylabel('$y$ [m]')
    theta = np.linspace(0, 2*np.pi, 100)
    axes.plot(2.2*np.cos(theta), 2.2*np.sin(theta), 'g--', alpha=0.3, label='workspace')
    axes.plot(x_e, y_e, 'b*', markersize=15, label='target')
    axes.legend()
    plt.show()

interactive(children=(FloatSlider(value=0.0, description='x_e', max=1.0, min=-1.0, step=0.2), FloatSlider(valu…